# Frost Days — exploration & statistiques descriptives

Ce notebook sert à **vérifier la qualité des données** et la **cohérence des résultats** du calcul des jours de gel (`TN <= 0 °C`).

Plan :
1. Référentiel des communes (complétude, coordonnées manquantes)
2. Données météo d'un département (stations, couverture, valeurs manquantes)
3. Distribution des températures minimales (`TN`) et plausibilité
4. Calcul d'un rapport de gel et visualisations
5. Comparaison de plusieurs communes (gradient altitude / littoral)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # rendre le package importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from frost_days import config, meteo, viz
from frost_days.communes import load_communes, find_commune
from frost_days.frost import compute_frost_report, evaluate_stations

pd.set_option('display.max_columns', 30)
print('Seuil valeurs manquantes :', config.MAX_MISSING_FRACTION)
print('Fenêtre de cache        :', config.CACHE_START, '->', config.CACHE_END)

## 1. Référentiel des communes

On vérifie le nombre de communes, la présence de coordonnées, et l'effet du dictionnaire de complétion (`coord_source == 'secours'`).

In [ ]:
communes = load_communes()
print('Nombre de communes :', len(communes))
print('Sans coordonnées   :', int((communes['lat'].isna() | communes['lon'].isna()).sum()))
print('Coord. via secours :', int((communes['coord_source'] == 'secours').sum()))
print('Nombre de départements :', communes['dep_code'].nunique())
communes.head()

In [ ]:
# Communes encore sans coordonnées (communes nouvelles récentes le plus souvent)
communes[communes['lat'].isna() | communes['lon'].isna()][['code_insee','nom','dep_code']]

## 2. Données météo d'un département

On charge un département (ici **05 — Hautes-Alpes**). Le premier appel télécharge et construit le cache Parquet ; ensuite c'est instantané.

In [ ]:
DEP = '05'
df = meteo.load_department(DEP)
print('Observations (2013-2024) :', f'{len(df):,}')
print('Stations distinctes      :', df['num_poste'].nunique())
print('Plage de dates           :', df['date'].min().date(), '->', df['date'].max().date())
df.head()

In [ ]:
# Qualité des stations sur la période 2014-2023
start, end = pd.Timestamp('2014-01-01'), pd.Timestamp('2023-12-31')
stations = evaluate_stations(df, start, end).sort_values('missing_fraction')
print('Stations avec données sur la période :', len(stations))
print('Stations sous le seuil de 35 %       :', int((stations['missing_fraction'] <= 0.35).sum()))
stations[['nom_usuel','alti','n_tn','missing_fraction']].head(10)

In [ ]:
# Distribution du taux de valeurs manquantes par station
fig, ax = plt.subplots(figsize=(7,3))
ax.hist(stations['missing_fraction']*100, bins=20, color='#4a78b5', edgecolor='white')
ax.axvline(35, color='crimson', ls='--', label='seuil 35 %')
ax.set_xlabel('Valeurs manquantes (%)'); ax.set_ylabel('Nb de stations'); ax.legend()
ax.set_title(f'Qualité des stations du département {DEP} (2014-2023)')
plt.tight_layout(); plt.show()

## 3. Distribution des températures minimales (TN)

Contrôle de plausibilité : les `TN` doivent rester dans une plage physique (≈ -40 °C à +30 °C). On visualise aussi la part de jours de gel.

In [ ]:
tn = df['tn'].dropna()
print(tn.describe())
print('\nPart de jours TN <= 0°C :', f'{(tn <= 0).mean():.1%}')
print('Valeurs aberrantes (<-40 ou >40) :', int(((tn < -40) | (tn > 40)).sum()))

fig, ax = plt.subplots(figsize=(7,3))
ax.hist(tn, bins=80, color='#4a78b5')
ax.axvline(0, color='crimson', ls='--', label='0 °C (seuil gel)')
ax.set_xlabel('TN (°C)'); ax.set_ylabel('Nb d\'observations'); ax.legend()
ax.set_title(f'Distribution des TN — département {DEP}')
plt.tight_layout(); plt.show()

## 4. Rapport de gel pour une commune

On calcule le rapport complet pour **Briançon (05)** sur 2014-2023 et on affiche les graphiques (Plotly).

In [ ]:
report = compute_frost_report('Briançon', '05', '2014-01-01', '2023-12-31', verbose=False)
print(report.summary_text())
report.per_year

In [ ]:
viz.fig_per_year(report).show()
viz.fig_day_of_year(report).show()
viz.fig_calendar_heatmap(report).show()

In [ ]:
# Jours calendaires les plus gélifs
report.day_of_year.sort_values(['freq_pct','n_frost'], ascending=False).head(10)

## 5. Comparaison de communes

Contrôle de cohérence : une commune d'altitude doit geler bien plus souvent qu'une commune littorale. On compare quelques cas (chaque département non encore en cache sera téléchargé au premier appel).

In [ ]:
cas = [('Briançon','05'), ('Nice','06'), ('Strasbourg','67')]
lignes = []
for nom, dep in cas:
    try:
        r = compute_frost_report(nom, dep, '2014-01-01', '2023-12-31', verbose=False)
        lignes.append({'commune': r.commune.nom, 'dep': dep, 'altitude_station': r.station_alti,
                       'jours_gel_total': r.total_frost_days, 'moyenne_par_an': round(r.avg_frost_days_per_year,1)})
    except Exception as e:
        lignes.append({'commune': nom, 'dep': dep, 'erreur': str(e)})
pd.DataFrame(lignes)

**Lecture attendue** : Briançon (alpine, ~1300 m) ≫ Strasbourg (continentale) ≫ Nice (littoral méditerranéen, gel rare). Si l'ordre est respecté, le calcul est cohérent.